## 1. Article Keyword Extraction (TF-IDF)
The first step in our pipeline is to identify the most significant words for each news article. We use the **TF-IDF (Term Frequency-Inverse Document Frequency)** algorithm to transform raw text into a weighted numerical representation.

* **Term Frequency (TF):** Measures how frequently a term occurs in a specific article.
* **Inverse Document Frequency (IDF):** Measures how important a term is across the entire collection. It helps filter out common words like "the" or "said" that don't carry specific meaning.
* **Implementation:** We clean the text by removing backslashes and whitespace, then use `TfidfVectorizer` to extract the top 10 highest-scoring words for every article.

In [1]:
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer

file_name = "us_newsarticle_August_1"

try:
    with open(file_name, 'r', encoding='utf-8') as file:
        content = file.read()

    raw_articles = re.split(r'\*+-----', content)

    processed_docs = []
    article_headers = []

    for entry in raw_articles:
        if entry.strip():
            # Corrected line
            clean_text = re.sub(r'\\', '', entry).strip()
            
            lines = [l for l in clean_text.split('\n') if l.strip()]
            if lines:
                article_headers.append(lines[0][:70])
                processed_docs.append(clean_text)

    vectorizer = TfidfVectorizer(stop_words='english', max_features=1000)
    tfidf_matrix = vectorizer.fit_transform(processed_docs)
    feature_names = vectorizer.get_feature_names_out()

    for i in range(len(processed_docs)):
        row = tfidf_matrix.getrow(i).toarray().flatten()
        top_indices = row.argsort()[-10:][::-1]
        
        print(f"Article {i+1}: {article_headers[i]}...")
        print("-" * 50)
        for idx in top_indices:
            if row[idx] > 0:
                print(f"{feature_names[idx]:<15} | Score: {row[idx]:.4f}")
        print("\n")

except FileNotFoundError:
    print(f"Error: Could not find the file '{file_name}'.")
except Exception as e:
    print(f"Error: {e}")

Article 1: -----Melania Trump's girl-on-girl photos from racy shoot revealed  ---...
--------------------------------------------------
melania         | Score: 0.5460
photo           | Score: 0.2730
basseville      | Score: 0.2409
fashion         | Score: 0.1927
shoot           | Score: 0.1927
beauty          | Score: 0.1712
pictures        | Score: 0.1712
featured        | Score: 0.1445
eriksson        | Score: 0.1445
lesbian         | Score: 0.1445


Article 2: Ex-Venezuelan narcotics officials face drug-trafficking charges  -----...
--------------------------------------------------
drug            | Score: 0.4664
brooklyn        | Score: 0.3071
molina          | Score: 0.2304
torres          | Score: 0.2304
narcotics       | Score: 0.2304
ex              | Score: 0.2304
charges         | Score: 0.2304
federal         | Score: 0.2140
trafficking     | Score: 0.2048
court           | Score: 0.2048


Article 3: Props used in iconic movies to hit the auction block  -----  August 1 ...

## 2. Social Media Matching via Jaccard Similarity
With the news keywords extracted, we now look for the most relevant tweets. Instead of simple keyword matching, we use **Jaccard Similarity** to find the degree of overlap between an article's profile and a tweet's content.

* **Jaccard Index:** This is defined as the size of the intersection divided by the size of the union of the word sets.
* **Formula:** $$J(A, B) = \frac{|A \cap B|}{|A \cup B|}$$
* **Normalization:** Tweets are heavily preprocessed to remove URLs, "RT" handles, and special characters to ensure we are comparing core concepts rather than noise.
* **Deduplication:** The pipeline filters out identical or near-duplicate tweets to ensure the results represent unique perspectives or reports.

In [4]:
import os
import re
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# --- SETTINGS ---
NEWS_FILE = "us_newsarticle_August_1"
TWEET_FOLDER = "Aug 1"
SIMILARITY_THRESHOLD = 0.05  # Jaccard scores are usually lower than Cosine
TOP_N_TWEETS = 10

def normalize_tweet(text, for_matching=False):
    """
    Cleans tweet text to identify duplicates and prepare for set matching.
    """
    # 1. Remove leading ID and asterisks
    text = re.sub(r'^\d+\*+', '', text)
    # 2. Remove 'RT @username:' 
    text = re.sub(r'^rt\s+@\w+[:\s]+', '', text, flags=re.IGNORECASE)
    # 3. Remove URLs
    text = re.sub(r'http\S+', '', text)
    # 4. Keep alphanumeric and lowercase
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text).lower()
    
    clean_list = text.split()
    if for_matching:
        return set(clean_list) # Return as a set for Jaccard math
    return " ".join(clean_list)

def get_jaccard_score(set_a, set_b):
    """Calculates Intersection over Union for two sets."""
    intersection = set_a.intersection(set_b)
    union = set_a.union(set_b)
    return len(intersection) / len(union) if union else 0.0

def get_news_data(file_path):
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            content = f.read()
        raw_chunks = re.split(r'\*+-----', content)
        docs, titles = [], []
        for chunk in raw_chunks:
            if chunk.strip():
                # Corrected regex for cleaning tags
                clean_text = re.sub(r'\\', '', chunk).strip()
                lines = [l for l in clean_text.split('\n') if l.strip()]
                if lines:
                    titles.append(lines[0][:70])
                    docs.append(clean_text)
        return titles, docs
    except FileNotFoundError:
        print(f"Error: {file_path} not found.")
        return [], []

def run_pipeline():
    # Step 1: Process News
    titles, docs = get_news_data(NEWS_FILE)
    if not docs: return

    # Step 2: Extract TF-IDF Keywords
    vectorizer = TfidfVectorizer(stop_words='english', max_features=1000)
    tfidf_matrix = vectorizer.fit_transform(docs)
    feature_names = vectorizer.get_feature_names_out()
    
    # Store article profiles as sets of keywords
    article_profiles = []
    for i in range(len(docs)):
        row = tfidf_matrix.getrow(i).toarray().flatten()
        top_indices = row.argsort()[-10:][::-1]
        keywords = {feature_names[idx] for idx in top_indices if row[idx] > 0}
        article_profiles.append(keywords)

    # Step 3: Match with Tweets using Jaccard
    results = {title: [] for title in titles}
    seen_normalized = {title: set() for title in titles}

    if not os.path.exists(TWEET_FOLDER):
        print(f"Folder '{TWEET_FOLDER}' not found.")
        return

    for filename in os.listdir(TWEET_FOLDER):
        file_path = os.path.join(TWEET_FOLDER, filename)
        if os.path.isfile(file_path):
            with open(file_path, 'r', encoding='utf-8') as f:
                for line in f:
                    original_tweet = line.strip()
                    if not original_tweet: continue
                    
                    # Normalize for matching (set) and deduplication (string)
                    tweet_set = normalize_tweet(original_tweet, for_matching=True)
                    norm_content = " ".join(sorted(list(tweet_set)))
                    
                    for art_idx, title in enumerate(titles):
                        # Calculate Jaccard Similarity
                        score = get_jaccard_score(article_profiles[art_idx], tweet_set)
                        
                        if score >= SIMILARITY_THRESHOLD and norm_content not in seen_normalized[title]:
                            if len(tweet_set) > 3:
                                results[title].append((original_tweet, round(score, 4)))
                                seen_normalized[title].add(norm_content)

    # Step 4: Display Results
    for title in titles:
        print(f"\n{'='*30}\nARTICLE: {title}\n{'='*30}")
        sorted_tweets = sorted(results[title], key=lambda x: x[1], reverse=True)[:TOP_N_TWEETS]
        if sorted_tweets:
            for i, (tweet, score) in enumerate(sorted_tweets):
                print(f"{i+1}. [{score}] {tweet}")
        else:
            print("No unique relevant tweets found.")

if __name__ == "__main__":
    run_pipeline()


ARTICLE: -----Melania Trump's girl-on-girl photos from racy shoot revealed  ---
1. [0.1667] 760226772977725440*****#fashion #celebs #d3320 #amandabynes candid #photo classy #sexy beauty #style #buzz
2. [0.1579] 760090755893334018*****#fashion #beauty #amandabynes 8x10 #photo picture hot #sexy candid 6 #buzz #celebs
3. [0.1538] 760170510575632384*****#fashion #beauty #sexy #lovely #style
4. [0.1429] 759913496196681728*****melissani lake. kefalonia, greece #pictures #beauty
5. [0.1429] 760035273635827712*****#actress #beauty original #jennifergarner signed #photo
6. [0.1429] 760122510939721728*****#pics #photos #photo #fotos #foto #pictures
7. [0.1333] 760189934401716224*****so chic guide is out! #fashion #beauty
8. [0.1333] 760261086574682112*****#lesbian orgy vids wwe divas pictures naked
9. [0.1333] 759997088700719105*****come sneak out this beauty photo #sneakpeaking #beauty
10. [0.1333] 760060301039562752*****#sex et porno pictures of lesbian fucking

ARTICLE: Ex-Venezuelan narcoti

### **Observation: High-Precision Matching for Zika Outbreak**
The results for the **Zika outbreak** article demonstrate the pipeline's effectiveness at capturing high-precision social media relevance. By combining TF-IDF keyword extraction with Jaccard Similarity, the system successfully isolated the most critical aspects of the news story.

#### **Why these results are excellent:**
* **Contextual Accuracy:** The matched tweets perfectly align with the core entities: **Zika**, **Miami**, **Travel Warnings**, and **Pregnant Women**.
* **Source Identification:** The algorithm successfully prioritized tweets mentioning the **CDC**, which was the primary authority cited in the original news report.
* **Geographic Specificity:** The results correctly identify the specific impact zones in **Florida** and **North Miami**, mirroring the level of detail found in the news article.
* **Strong Similarity Scores:** With Jaccard scores reaching as high as **0.3333**, these results represent a very strong linguistic overlap, proving that the top 10 TF-IDF keywords are highly representative of the actual public discourse.

### **Analysis: Low-Precision Matching for "Prankster Cops" Article**
In contrast to the Zika article, the results for the **"Prankster cops hand out ice cream cones"** story are largely irrelevant. While the algorithm is functioning correctly from a mathematical standpoint, this example highlights a classic challenge in Natural Language Processing (NLP).

#### **Why the results are irrelevant:**
* **Semantic Ambiguity:** The top TF-IDF keywords for this article included general terms like **"cops"** and **"ice cream"**. On Twitter, "ice cream" is frequently used in unrelated contexts, such as pop culture fandoms (e.g., `#pushawardskathniels`) or food reviews, which diluted the search results.
* **Keyword Overlap without Context:** The Jaccard Similarity identified tweets containing the words "cops" or "ice cream," but these tweets were about bank robberies, assaults, or unrelated police shootings. The algorithm matched the **words** but failed to understand the **whimsical intent** of the original story.
* **Co-occurrence Noise:** Because "ice cream" and "cops" are common nouns, they appear in thousands of unrelated tweets. Without a unique "anchor" word (like a specific location or a rare medical term like "Zika"), the Jaccard score cannot distinguish between a lighthearted news story and a random tweet about a snack or a crime.

#### **Conclusion:**
This highlights the need for **Entity Recognition** or **Sentiment Analysis** in more advanced pipelines to differentiate between "Ice Cream" as a prank and "Ice Cream" as a generic food mention.

### **Observation: Broad Context Matching in Political Articles**
The results for the **"Warren Buffett joins the billionaires-for-Hillary"** article illustrate a "keyword dilution" effect. While the first result is a perfect match, the remaining nine are generic political tweets.

#### **Why only one result was context-specific:**
* **Dominance of High-Frequency Names:** The TF-IDF extraction identified **"Clinton"** and **"Trump"** as top keywords. Because these names appeared in a massive volume of tweets on August 1st, the Jaccard Similarity pulled in any tweet discussing the general election, even if it didn't mention Warren Buffett.
* **Insufficient Weight for Unique Entities:** Unique but less frequent terms like **"Buffett"** or **"Omaha"** had to compete with the overwhelming signal of "Clinton" and "Trump." If a tweet contained "Clinton" and "Trump" but not "Buffett," it still achieved a high enough similarity score to displace more relevant, specific content.

#### **How to Improve Accuracy:**
To refine the results and move beyond broad political noise, we could implement the following:

1. **Named Entity Recognition (NER):** Use a library like `spaCy` to specifically identify and prioritize **Persons** (Warren Buffett) and **Organizations** (Billionaires-for-Hillary). We could require that specific "Key Entities" must be present in the tweet for it to be considered.
2. **Dynamic Weighting:** Increase the importance of keywords that are rare across the entire tweet dataset. If "Buffett" appears in only 0.1% of tweets while "Clinton" appears in 20%, "Buffett" should be given a much higher weight in the similarity calculation.
3. **Bigram/Trigram Analysis:** Instead of looking at single words (unigrams), we could look for phrases like **"Warren Buffett"** or **"joins Hillary."** This ensures the "action" of the article is captured, not just the names.
4. **Sentiment or Topic Clustering:** Use LDA (Latent Dirichlet Allocation) to ensure the tweet belongs to the "Economic/Endorsement" topic rather than "General Campaign News" or "Email Scandals."

# Final Project Summary and Conclusion

This project implemented an NLP pipeline designed to connect traditional news reporting with real-time social media discourse. By utilizing **TF-IDF Keyword Extraction** and **Jaccard Similarity**, we analyzed how news stories propagate on Twitter and identified the strengths and limitations of token-based matching.

---

### 1. Performance Analysis: The Good, The Bad, and The Noisy

The effectiveness of the pipeline depended heavily on the **linguistic uniqueness** of the news topic:

* **The Success (High Specificity):** The **Zika Outbreak** article yielded excellent results. Because it contained specific, rare tokens like "Zika" and "CDC," the Jaccard Similarity was able to isolate highly relevant tweets with precision.
* **The Failure (Semantic Ambiguity):** The **"Prankster Cops"** article failed because its keywords—"cops" and "ice cream"—are common nouns used in thousands of unrelated contexts. The model matched the *words* but missed the *intent*, retrieving news about crime and general food tweets instead of the specific prank.
* **The Noise (Broad Context):** The **Warren Buffett/Hillary Clinton** article suffered from keyword flooding. Since "Trump" and "Clinton" were the most frequent terms on Twitter during this period, specific news about Buffett was often buried under a mountain of generic political debate.

---

### 2. Technical Evaluation

| Technique | Role | Evaluation |
| :--- | :--- | :--- |
| **TF-IDF** | Distills articles into 10 "anchor" words. | Strong at finding descriptive terms, but occasionally picks up generic high-frequency words. |
| **Jaccard Similarity** | Measures overlap between keyword sets. | Transparent and fast, but lacks "semantic depth"—it cannot distinguish between different meanings of the same word. |
| **Deduplication** | Filters out repetitive content. | Successful at removing exact retweets, ensuring a diverse list of top results. |

---

### 3. Proposed Enhancements for Future Iterations

To improve the precision of the matches, the following strategies should be considered:

1.  **Named Entity Recognition (NER):** Use libraries like `spaCy` to give higher weight to **People** (e.g., Warren Buffett) and **Organizations** (e.g., CDC) while ignoring generic nouns.
2.  **Sentence Embeddings (BERT):** Replace Jaccard Similarity with **Cosine Similarity** using BERT or RoBERTa. This would allow the model to understand that "handing out ice cream" is a positive, whimsical act, distinct from "reporting a crime."
3.  **Bigram Analysis:** Look for word pairs (e.g., "travel warning") rather than single words to preserve more context during the matching phase.

### **Final Conclusion**
This pipeline serves as a powerful baseline for information retrieval. It demonstrates that while statistical methods like TF-IDF are excellent for broad categorization, interpreting the nuance of human conversation on social media requires additional layers of contextual and semantic logic.